# Choosing `space_time_ratio` for TAM3C2

TAM3C2 aggregates point-cloud neighbourhoods **jointly in space and time** before computing the M3C2 distance. The temporal half of that aggregation is shaped by a single unitless parameter — `space_time_ratio`.

This notebook explains what the parameter does and then runs an automated sweep to pick a sensible value. The basic idea: larger `space_time_ratio` means more temporal averaging and a smaller LoD; but if it is pushed too far, real surface change starts to leak into the within-neighbourhood spread and inflates the M3C2 uncertainty in a misleading way. So we want the largest value that does *not* visibly inflate the spread on areas we know to be stable.

## 1. Background — why a space-time ratio?

Inside one corepoint neighbourhood every point has a spatial offset to the corepoint (in metres) and a temporal offset to the reference time (in seconds). These two quantities have different units, so before they can be combined into a single weight they are each rescaled by their own window size — the spatial offset is divided by the cylinder radius, and the temporal offset is divided by the temporal half-window `|t_target - t_ref| · max_window_ratio`.

Even after this rescaling, there is still a choice to make: how much do we *trust* a neighbour that is one window away in time compared to one that is one window away in space? `space_time_ratio` is exactly that knob. A Gaussian weight (controlled by `sigma_ratio`) is applied to the rescaled spatial offset and to the rescaled temporal offset divided by `space_time_ratio`. So the larger `space_time_ratio` is, the slower the temporal weight decays and the more distant epochs end up contributing.

Interpretation (with `sigma_ratio = 1` for orientation):

| `space_time_ratio` | Meaning |
|---|---|
| **= 1** | symmetric: a 1-window time offset is penalised the same as a 1-radius space offset |
| **> 1** | trust time more: temporal weight decays slower → more epochs aggregate → smoother / lower LoD |
| **< 1** | trust space more: temporal weight decays faster → fewer epochs aggregate → closer to single-epoch M3C2 |

Regardless of `space_time_ratio`, the hard cap `max_window_ratio` and the `required_points` check still apply — `space_time_ratio` only reshapes the *soft* decay inside that window.

## 2. How do we pick `space_time_ratio`?

We use a simple, intuitive idea: **look at parts of the scene that we know are stable, and see what happens to the M3C2 within-neighbourhood spread as we change the parameter**.

The within-neighbourhood spread that M3C2 returns (in the `uncertainties.spread1` / `spread2` fields) has two contributions: the *real* geometric roughness of the surface inside the neighbourhood, which is exactly what M3C2's LoD formula is designed to account for; and *contamination* from any actual surface motion that happens within the aggregation window, which gets mixed into the spread when we aggregate across epochs. On a corepoint that does not change over the whole time series, the second contribution should stay close to zero — unless we make the temporal weighting so generous that microscopic drift starts to creep in.

So the recipe is:

1. **Pick a stable subset of corepoints.** A corepoint is treated as stable if its maximum absolute change across all targets is below a small threshold (we reuse the `obc_height_threshold`-style value).
2. **For each candidate `space_time_ratio`**, run the full TAM3C2 pipeline and read off the mean of `spread1` and `spread2` over the stable corepoints and over all targets. Call this the stable-spread for that ratio.
3. **Use the smallest candidate ratio as the baseline.** It does the least temporal smoothing of the candidates we consider, so its stable-spread is essentially the geometric-noise floor.
4. **Compare each ratio's stable-spread against this baseline.** A ratio that produces almost the same stable-spread as the baseline is doing temporal aggregation without polluting the spread; a ratio whose stable-spread is much larger than the baseline is pulling in real surface change.
5. **Pick the largest ratio whose stable-spread stays within a tolerance** of the baseline (default: at most 10 % above the baseline). That buys the most LoD reduction we can get for an essentially-unchanged stable spread.

The helper `estimate_space_time_ratio` does steps 1–5 and also returns the per-ratio means of LoD95 and effective sample count so we can plot what we gain (a smaller detection threshold, more effective samples) against what we pay (a possibly larger stable-spread).

---
## 3. Algorithm — what `estimate_space_time_ratio` does step by step

Inputs and their roles:

| Argument | Meaning |
|---|---|
| `candidate_ratios` | the grid we sweep; default `(0.5, 1.0, 2.0, 4.0)` |
| `tolerance` | how much the stable-spread is allowed to grow relative to the baseline; default `0.1` (i.e. 10%) |
| `stable_threshold` | a corepoint is *stable* if its maximum absolute change across all targets is below this value (in metres). A typical choice is the same value as `obc_height_threshold`. |
| `tam_kwargs` | every other TAM3C2 / M3C2 parameter (cyl_radius, max_distance, ...). Whatever you would pass to the constructor in the main pipeline. |

Procedure:

1. For every candidate ratio, build a fresh `TAM3C2(space_time_ratio=r, **tam_kwargs)` and call `calculate_distances(ref, tgt)` for every target. Collect the per-target `distances`, `spread1`, `spread2`, `lod95` and `num_samples` arrays.
2. Mark stable corepoints — those whose maximum absolute distance across all targets is below `stable_threshold`.
3. For every candidate ratio, compute the mean of `spread1` and `spread2` over the stable corepoints and over all targets — that is the stable-spread for that ratio. Take the smallest candidate ratio as the baseline.
4. Return the largest ratio whose stable-spread is at most `(1 + tolerance)` times the baseline. If none qualifies, return the ratio with the smallest stable-spread and flag `constraint_satisfied=False`.

The returned `dict` also exposes the full per-ratio report so you can plot:

- the stable-spread (relative to the baseline) as a function of ratio — should stay flat as long as we have not started pulling in real motion;
- mean LoD95 as a function of ratio — should fall as the ratio grows;
- mean effective sample count as a function of ratio — should rise as the ratio grows.


## 4. Setup

We reuse the same data and parameters as the main `test_S1_uls_tam3c2.ipynb` so the chosen $r_{st}$ transfers directly.

In [ ]:
from datetime import datetime
import os, numpy as np, matplotlib.pyplot as plt
import py4dgeo
from py4dgeo import (
    TAM3C2, Weighting,
    read_epochs_from_folder, extract_reference_and_others, sample_corepoints,
    sweep_space_time_ratio, estimate_space_time_ratio,
)

In [ ]:
data_path = r'C:\rsa\research_proj\time_aware_M3C2\test\salt_marsh\Rennes_database_COSMA_downsampled'

reference_timestamp = datetime(2010, 10, 4)
max_epochs_after_reference = 120

# Corepoint sampling
corepoint_voxel_size = 1.5

# TAM3C2 / M3C2 parameters (same as the main notebook)
normal_radii      = [1.0]
max_window_ratio  = [0.1, 0.2, 0.3, 0.4, 0.5]
required_points   = 10
cyl_radius        = 1.0
max_distance      = 5.0
registration_error = 0.02
sigma_ratio       = 1.0
weighting         = Weighting.GAUSSIAN

# Stable-corepoint threshold (use the same value as obc_height_threshold)
stable_threshold = 0.1

# Ratio sweep settings  (widened to expose any non-monotone behaviour)
candidate_ratios = (0.125, 0.25, 0.5, 1.0, 2.0, 4.0, 8.0)
tolerance        = 0.1  # 10 % inflation budget

# To keep the sweep fast we evaluate the criterion on a subset of targets
n_sweep_targets = 30

In [ ]:
epochs = read_epochs_from_folder(data_path)
if max_epochs_after_reference is not None:
    sorted_eps = sorted(epochs, key=lambda e: e.timestamp)
    ref_idx = next(i for i, e in enumerate(sorted_eps) if e.timestamp == reference_timestamp)
    epochs = sorted_eps[: ref_idx + 1 + max_epochs_after_reference]

reference_epoch, other_epochs = extract_reference_and_others(epochs, reference_timestamp)
corepoints = sample_corepoints(reference_epoch, method='voxel', voxel_size=corepoint_voxel_size)
print(f"epochs: {len(epochs)}  |  reference: {reference_epoch.timestamp}  |  corepoints: {len(corepoints):,}")

# Evenly spaced subset of targets for the sweep
idx = np.linspace(0, len(other_epochs) - 1, num=min(n_sweep_targets, len(other_epochs)), dtype=int)
target_subset = [other_epochs[i] for i in idx]
print(f"sweep targets: {len(target_subset)}")

## 5. Run the sweep + selection

Everything that would normally be passed to `TAM3C2(...)` goes into `tam_kwargs` (everything except `space_time_ratio`, `epochs_timeseries` and `corepoints`, which `estimate_space_time_ratio` controls itself).


In [ ]:
tam_kwargs = dict(
    max_window_ratio=max_window_ratio,
    normal_radii=normal_radii,
    required_points=required_points,
    weighting=weighting,
    sigma_ratio=sigma_ratio,
    cyl_radius=cyl_radius,
    max_distance=max_distance,
    registration_error=registration_error,
)

result = estimate_space_time_ratio(
    epochs_timeseries=epochs,
    reference_epoch=reference_epoch,
    target_epochs=target_subset,
    corepoints=corepoints,
    tam_kwargs=tam_kwargs,
    candidate_ratios=candidate_ratios,
    tolerance=tolerance,
    stable_threshold=stable_threshold,
)

print(f"best space_time_ratio = {result['best_ratio']}")
print(f"constraint satisfied  = {result['constraint_satisfied']}")
print(f"baseline ratio        = {result['baseline_ratio']}  (used to define rho = 1)")
print(f"#stable corepoints    = {int(result['stable_mask'].sum())} / {len(corepoints)}")

In [ ]:
# Per-ratio report
print(f"{'r_st':>6}  {'rho':>8}  {'mean LoD95 [m]':>16}  {'mean N_eff':>12}")
for r in result['report']:
    marker = '  <-- best' if r['space_time_ratio'] == result['best_ratio'] else ''
    print(f"{r['space_time_ratio']:>6.2f}  {r['rho']:>8.4f}  {r['mean_lod95']:>16.4f}  {r['mean_num_samples']:>12.1f}{marker}")

## 6. Diagnostic plots

Three curves tell the whole story:

1. **Relative stable-spread** — the mean within-neighbourhood spread on the stable corepoints, normalised by the baseline ratio. We want this to stay below `1 + tolerance`.
2. **Mean LoD95** — should drop as `space_time_ratio` grows (the *benefit* of aggregation).
3. **Mean effective sample count** — should rise as `space_time_ratio` grows (the *evidence* that we are actually aggregating).


In [ ]:
ratios   = np.array([r['space_time_ratio']  for r in result['report']])
rhos     = np.array([r['rho']               for r in result['report']])
lods     = np.array([r['mean_lod95']        for r in result['report']])
neffs    = np.array([r['mean_num_samples']  for r in result['report']])
best_r   = result['best_ratio']

fig, axs = plt.subplots(1, 3, figsize=(15, 4))

axs[0].plot(ratios, rhos, 'o-')
axs[0].axhline(1.0 + tolerance, color='r', ls='--', label=f'1 + tolerance = {1+tolerance:.2f}')
#axs[0].axvline(best_r,          color='g', ls='--', label=f'selected r_st = {best_r}')
axs[0].set_xscale('log')
axs[0].set_xlabel('space_time_ratio  (log)')
axs[0].set_ylabel(r'spread inflation $\rho$')
axs[0].set_title('Stable-corepoint spread inflation')
axs[0].grid(alpha=0.3)
axs[0].legend()

axs[1].plot(ratios, lods, 'o-')
#axs[1].axvline(best_r, color='g', ls='--')
axs[1].set_xscale('log')
axs[1].set_xlabel('space_time_ratio  (log)')
axs[1].set_ylabel('mean LoD95 [m]')
axs[1].set_title('Detection threshold')
axs[1].grid(alpha=0.3)

axs[2].plot(ratios, neffs, 'o-')
# axs[2].axvline(best_r, color='g', ls='--')
axs[2].set_xscale('log')
axs[2].set_xlabel('space_time_ratio  (log)')
axs[2].set_ylabel(r'mean $N_{eff}$')
axs[2].set_title('Effective sample count')
axs[2].grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Spatial distribution of the stable corepoints used in rho(r_st)
stable = result['stable_mask']
plt.figure(figsize=(9, 5))
plt.scatter(corepoints[~stable, 0], corepoints[~stable, 1], s=5, c='lightgrey', label='changing')
plt.scatter(corepoints[ stable, 0], corepoints[ stable, 1], s=8, c='tab:blue',  label='stable (baseline)')
plt.gca().set_aspect('equal')
plt.xlabel('X [m]'); plt.ylabel('Y [m]')
plt.title(f'Stable corepoints ({int(stable.sum())}/{len(corepoints)}) — used for rho')
plt.legend(); plt.tight_layout(); plt.show()

## 7. Conclusion and recommended value

Based on the 1D sweep in §5 and the 2D joint analysis in §8, we can draw a clear conclusion for this dataset.

**Key observations**

1.  **Low sensitivity**: The stable-spread is barely affected by `space_time_ratio`. Across all tested `max_window_ratio` values, it remains well within the tolerance band around the baseline. This means that, on the parts of the scene we know to be stable, the temporal weighting anisotropy has a negligible impact on the within-neighbourhood spread.
2.  **Diminishing returns**: The detection threshold LoD95 improves as `space_time_ratio` increases, but the improvement becomes marginal beyond 1. The most significant gains are achieved when moving from a very small ratio to 1.
3.  **Robustness**: The 2D analysis confirms that this low sensitivity holds regardless of the hard temporal cap set by `max_window_ratio`.

**Recommendation**

**A default value of `space_time_ratio = 1.0` is recommended.**

This choice is justified because:
*   It captures most of the available LoD improvement while keeping the stable-spread essentially unchanged.
*   It is a physically intuitive, **isotropic** weighting kernel where space and time are treated equally — avoiding the need to defend a more complex anisotropic model that the data does not support.
*   It simplifies the model by removing a parameter that would otherwise need fine-tuning, making TAM3C2 more robust and easier to apply in practice.


## 8. Joint analysis of `max_window_ratio` and `space_time_ratio`

The 1D sweep above lives **inside a fixed hard temporal window**, set by `max_window_ratio`. Inside that window, `space_time_ratio` only *reshapes* the soft Gaussian decay across epochs — it cannot bring in new epochs, only reweight the ones already there. So if the surface drift inside this window is tiny, the stable-spread curve will look flat and "larger `space_time_ratio` is always better" is the genuine answer for that window.

To **separate** the two effects we now sweep both axes jointly:

* `max_window_ratio` — the **hard cap** on aggregated time (the real bias-variance lever).
* `space_time_ratio` — the **soft kernel anisotropy** inside that cap (what this notebook is about).

For every cell on the `(max_window_ratio, space_time_ratio)` grid we compute the stable-spread and the mean LoD95 on the **same stable corepoint set** (taken from the §5 baseline run so that all cells are directly comparable).

**What we expect to see**

1. As `max_window_ratio` grows (rows), the stable-spread should eventually start to grow — that is where real drift enters the window.
2. Inside any *single* row, the stable-spread as a function of `space_time_ratio` tells us whether the soft weighting matters at that hard-cap setting:
   * a flat curve ⇒ `space_time_ratio` does not matter at that window → fix it to 1 in that regime;
   * a rising curve ⇒ `space_time_ratio` matters → the §2 selection rule has real teeth.

**Reading the result for `space_time_ratio`**

If the stable-spread stays flat across *all* hard-cap rows we use in practice, then `space_time_ratio = 1` is justified as a **principled default**: the data tells us the soft anisotropy is irrelevant in our operating regime, and a symmetric kernel is the most defensible choice.


In [ ]:
# ---- Path B grid configuration -----------------------------------------
# Reuse the stable mask from section 5 so every (mwr, r_st) cell is
# evaluated on the *same* corepoints. Comparable across rows and columns.
stable = result['stable_mask']
print(f"stable corepoints reused from section 5: {int(stable.sum())} / {len(corepoints)}")

# Each entry is a *list* because TAM3C2's `max_window_ratio` expects
# a list (multi-scale window selection). Single-element list = fixed window.
mwr_grid = [[0.1], [0.2], [0.3], [0.4], [0.6]]
rst_grid = candidate_ratios  # same widened set as the 1D sweep

print(f"grid size: {len(mwr_grid)} x {len(rst_grid)} = "
      f"{len(mwr_grid) * len(rst_grid)} TAM3C2 calls "
      f"x {len(target_subset)} targets")

In [ ]:
# ---- Run the 2D sweep --------------------------------------------------
# For every max_window_ratio row, sweep all candidate r_st values.
# Baseline for rho is the smallest r_st *within that same row*, so each
# row is normalised to its own geometric-noise floor.
rho_mat  = np.full((len(mwr_grid), len(rst_grid)), np.nan)
lod_mat  = np.full_like(rho_mat, np.nan)
neff_mat = np.full_like(rho_mat, np.nan)
spread_mat = np.full_like(rho_mat, np.nan)

def _mean_spread(entry, mask):
    return float(0.5 * (np.nanmean(entry['spread1'][mask])
                        + np.nanmean(entry['spread2'][mask])))

for i, mwr in enumerate(mwr_grid):
    kw = dict(tam_kwargs)
    kw['max_window_ratio'] = mwr
    sw = sweep_space_time_ratio(
        rst_grid,
        epochs_timeseries=epochs,
        reference_epoch=reference_epoch,
        target_epochs=target_subset,
        corepoints=corepoints,
        tam_kwargs=kw,
    )
    s_base = _mean_spread(sw[0], stable)
    for j, entry in enumerate(sw):
        s_j = _mean_spread(entry, stable)
        spread_mat[i, j] = s_j
        rho_mat[i, j]    = s_j / s_base if s_base > 0 else np.nan
        lod_mat[i, j]    = float(np.nanmean(entry['lod95']))
        neff_mat[i, j]   = float(0.5 * (np.nanmean(entry['num_samples1'])
                                        + np.nanmean(entry['num_samples2'])))
    print(f"  mwr={mwr}  done  (baseline spread={s_base:.4f} m)")

print("done.")

In [ ]:
# ---- Heatmaps:  rho(r_st, mwr)   and   mean LoD95(r_st, mwr) -----------
mwr_labels = [str(m[0]) if len(m) == 1 else f"{min(m)}-{max(m)}" for m in mwr_grid]
rst_labels = [f"{r:g}" for r in rst_grid]

fig, axs = plt.subplots(1, 2, figsize=(14, 4.5))

im0 = axs[0].imshow(rho_mat, aspect='auto', cmap='RdYlBu_r',
                    vmin=0.9, vmax=max(1.5, np.nanmax(rho_mat)))
axs[0].set_xticks(range(len(rst_labels)), rst_labels)
axs[0].set_yticks(range(len(mwr_labels)), mwr_labels)
axs[0].set_xlabel('space_time_ratio  $r_{st}$')
axs[0].set_ylabel('max_window_ratio')
axs[0].set_title(r'spread inflation $\rho(r_{st},\,W)$')
# annotate cells with rho values; mark cells above 1+tol with a star
for i in range(rho_mat.shape[0]):
    for j in range(rho_mat.shape[1]):
        v = rho_mat[i, j]
        if np.isnan(v):
            continue
        mark = '*' if v > 1.0 + tolerance else ''
        axs[0].text(j, i, f"{v:.2f}{mark}", ha='center', va='center',
                    color='black', fontsize=8)
plt.colorbar(im0, ax=axs[0], label=r'$\rho$')

im1 = axs[1].imshow(lod_mat, aspect='auto', cmap='viridis')
axs[1].set_xticks(range(len(rst_labels)), rst_labels)
axs[1].set_yticks(range(len(mwr_labels)), mwr_labels)
axs[1].set_xlabel('space_time_ratio  $r_{st}$')
axs[1].set_ylabel('max_window_ratio')
axs[1].set_title('mean LoD95 [m]')
for i in range(lod_mat.shape[0]):
    for j in range(lod_mat.shape[1]):
        v = lod_mat[i, j]
        if np.isnan(v):
            continue
        axs[1].text(j, i, f"{v:.3f}", ha='center', va='center',
                    color='white', fontsize=8)
plt.colorbar(im1, ax=axs[1], label='LoD95 [m]')

plt.suptitle(f'Joint sweep — {int(stable.sum())} stable corepoints, '
             f'{len(target_subset)} targets  (* = $\\rho > 1+\\tau$)')
plt.tight_layout()
plt.show()

In [ ]:
# ---- Overlay curves: one rho(r_st) per max_window_ratio ----------------
fig, axs = plt.subplots(1, 2, figsize=(14, 4.5))

for i, label in enumerate(mwr_labels):
    axs[0].plot(rst_grid, rho_mat[i], 'o-', label=f'mwr={label}')
    axs[1].plot(rst_grid, lod_mat[i], 'o-', label=f'mwr={label}')

axs[0].axhline(1.0 + tolerance, color='r', ls='--', alpha=0.7,
               label=f'1 + tolerance = {1+tolerance:.2f}')
axs[0].set_xscale('log')
axs[0].set_xlabel(r'$r_{st}$  (log)')
axs[0].set_ylabel(r'spread inflation $\rho$')
axs[0].set_title(r'$\rho(r_{st})$  for each max_window_ratio')
axs[0].grid(alpha=0.3)
axs[0].legend(fontsize=8)

axs[1].set_xscale('log')
axs[1].set_xlabel(r'$r_{st}$  (log)')
axs[1].set_ylabel('mean LoD95 [m]')
axs[1].set_title(r'LoD95$(r_{st})$  for each max_window_ratio')
axs[1].grid(alpha=0.3)
axs[1].legend(fontsize=8)

plt.tight_layout()
plt.show()

### 8.1 How to read the 2D result

* **Row direction (`max_window_ratio`)** — the *hard cap* on aggregated time. Moving down should eventually inflate the stable-spread (drift enters the window) and lower LoD95 (more samples contribute).
* **Column direction (`space_time_ratio`)** — the *soft anisotropy* inside the cap. A flat column at a given row means `space_time_ratio` has no effect at that hard cap. A rising column means it does — at the cost of a higher stable-spread.

**Possible patterns and what they imply for `space_time_ratio`**

1. **All rows flat** → `space_time_ratio` is irrelevant in the operating range of `max_window_ratio` we care about. Justify `space_time_ratio = 1` as the symmetric / parsimonious default.
2. **Rows flat at small `max_window_ratio`, rising at large `max_window_ratio`** → the soft anisotropy only matters once the hard cap is wide enough to admit drifting epochs. As long as the production pipeline uses the small-cap row, `space_time_ratio = 1` is safe.
3. **Rows rise even at small `max_window_ratio`** → the soft kernel really does matter. Pick the `space_time_ratio` that minimises LoD subject to the stable-spread staying within tolerance, in the row corresponding to your production `max_window_ratio`.

In all three cases the 2D heatmap is the evidence to put in the slides: it justifies the chosen `space_time_ratio` in light of the chosen `max_window_ratio`, rather than picking `space_time_ratio` in isolation.
